## Creation of the exposure datasets

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import xarray as xr
import shapely
from tqdm import tqdm

In [2]:
path = os.path.join('..','..','Data','INSEE','Filosofi2017_carreaux_1km_gpkg', 'Filosofi2017_carreaux_1km_met.gpkg')

In [26]:
pop = gpd.read_file(os.path.join('..','..','Data','INSEE','Filosofi2019_carreaux_200m_gpkg', 'carreaux_200m_met.gpkg'))


In [38]:
communes_littorales = gpd.read_file("../data/communes_littorales/communes_littorales.shp")
communes_littorales['INSEE_COM'] = communes_littorales['INSEE_COM'].astype(str).str.zfill(5)
communes_littorales['area'] = communes_littorales.geometry.area

comm_decret_2022 = pd.read_csv("../data/communes_littorales/comm_decret_2022.txt", header=None).squeeze().tolist()
comm_decret_2022 = [str(x).zfill(5) for x in comm_decret_2022]  # Ensure all codes are 5 digits
comm_decret_2023 = pd.read_csv("../data/communes_littorales/comm_decret_2023.txt", header=None).squeeze().tolist()
comm_decret_2024 = pd.read_csv("../data/communes_littorales/comm_decret_2024.txt", header=None).squeeze().tolist()

communes_littorales["decret_2022"] = communes_littorales["INSEE_COM"].isin(comm_decret_2022)
communes_littorales["decret_2023"] = communes_littorales["INSEE_COM"].isin(comm_decret_2023)
communes_littorales["decret_2024"] = communes_littorales["INSEE_COM"].isin(comm_decret_2024)

In [32]:
pop['lcog_geo'] = pop['lcog_geo'].astype(str)
pop = pop[pop.lcog_geo.isin(communes_littorales['INSEE_COM'].tolist())]

In [49]:
pop.columns

Index(['idcar_200m', 'idcar_1km', 'idcar_nat', 'i_est_200', 'i_est_1km',
       'lcog_geo', 'ind', 'men', 'men_pauv', 'men_1ind', 'men_5ind',
       'men_prop', 'men_fmp', 'ind_snv', 'men_surf', 'men_coll', 'men_mais',
       'log_av45', 'log_45_70', 'log_70_90', 'log_ap90', 'log_inc', 'log_soc',
       'ind_0_3', 'ind_4_5', 'ind_6_10', 'ind_11_17', 'ind_18_24', 'ind_25_39',
       'ind_40_54', 'ind_55_64', 'ind_65_79', 'ind_80p', 'ind_inc', 'geometry',
       'area'],
      dtype='object')

In [39]:
phma_comm = gpd.read_file("../data/phma1m_comm_dissolved.shp")
phma_comm['area_comm'] = phma_comm.apply(lambda x: communes_littorales[communes_littorales['INSEE_COM'] == x['INSEE_COM']]['area'].values[0], axis=1)
phma_comm['decret_2024'] = phma_comm.apply(lambda x: communes_littorales[communes_littorales['INSEE_COM'] == x['INSEE_COM']]['decret_2024'].values[0], axis=1)
phma_comm['ratio'] = phma_comm['area_phma'] / phma_comm['area_comm']
phma_comm.sort_values('ratio', ascending=False, inplace=True)

In [36]:
phma_comm

,INSEE_COM,code_insee,surf,Loi-litt,ID,STATUT,NOM_COM,INSEE_ARR,NOM_DEP,INSEE_DEP,NOM_REG,INSEE_REG,CODE_EPCI,NOM_COM_M,POPULATION,area_phma,geometry
0,06004,06004,251.3,Mer,BDCSURCO0000000009613085,Commune simple,Antibes,1,ALPES-MARITIMES,06,PROVENCE-ALPES-COTE D'AZUR,93,240600585,ANTIBES,75731,0.244136,"MULTIPOLYGON (((1033450.500 6284500.500, 10334..."
1,06011,06011,109.9,Mer,BDCSURCO0000000009612371,Commune simple,Beaulieu-sur-Mer,2,ALPES-MARITIMES,06,PROVENCE-ALPES-COTE D'AZUR,93,200030195,BEAULIEU-SUR-MER,3745,0.038956,"MULTIPOLYGON (((1048897.672 6298780.885, 10488..."
2,06027,06027,6130.1,Mer,BDCSURCO0000000009612605,Commune simple,Cagnes-sur-Mer,1,ALPES-MARITIMES,06,PROVENCE-ALPES-COTE D'AZUR,93,200030195,CAGNES-SUR-MER,47811,0.050969,"MULTIPOLYGON (((1034170.500 6291592.500, 10341..."
3,06029,06029,4446.1,Mer,BDCSURCO0000000009613330,Commune simple,Cannes,1,ALPES-MARITIMES,06,PROVENCE-ALPES-COTE D'AZUR,93,200039915,CANNES,73744,0.342104,"MULTIPOLYGON (((1028231.200 6275806.200, 10282..."
4,06032,06032,2686.9,Mer,BDCSURCO0000000009612370,Commune simple,Cap-d'Ail,2,ALPES-MARITIMES,06,PROVENCE-ALPES-COTE D'AZUR,93,200030195,CAP-D'AIL,4711,0.041832,"MULTIPOLYGON (((1054427.788 6301087.150, 10544..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,85278,85114,173359.1,Mer,BDCSURCO0000000009602513,Commune simple,Saint-Vincent-sur-Jard,3,VENDEE,85,PAYS DE LA LOIRE,52,200071900,SAINT-VINCENT-SUR-JARD,1314,1.760384,"MULTIPOLYGON (((350629.174 6599567.835, 350632..."
925,85288,85114,4488130.8,Mer,BDCSURCO0000000009602514,Commune simple,Talmont-Saint-Hilaire,3,VENDEE,85,PAYS DE LA LOIRE,52,200071900,TALMONT-SAINT-HILAIRE,7286,7.999778,"MULTIPOLYGON (((345398.493 6602560.808, 345395..."
926,85294,85001,4864572.9,Mer,BDCSURCO0000002000981230,Commune simple,La Tranche-sur-Mer,3,VENDEE,85,PAYS DE LA LOIRE,52,200073260,LA TRANCHE-SUR-MER,2818,11.839511,"MULTIPOLYGON (((364265.500 6592563.500, 364271..."
927,85297,85049,49414127.0,Mer,BDCSURCO0000000009602962,Commune simple,Triaize,1,VENDEE,85,PAYS DE LA LOIRE,52,200073260,TRIAIZE,1062,56.449470,"MULTIPOLYGON (((376813.182 6588201.455, 376813..."


In [ ]:
#make a test on the first 10 communes
phma_test = phma_comm.head(10)

In [45]:
phma_comm.shape

(929, 20)

In [64]:
pop_mean_com = pop[['ind', 'ind_65_79', 'ind_80p', 'ind_snv', 'lcog_geo']].groupby('lcog_geo').sum().reset_index()
pop_mean_com['ind_snv'] = pop_mean_com['ind_snv']/ pop_mean_com['ind']

In [ ]:
pop['area'] = pop.geometry.area
pop_litt = gpd.sjoin(pop[['geometry', 'ind', 'ind_65_79', 'ind_80p', 'ind_snv', 'lcog_geo', 'area']], phma_comm, how='inner', predicate='intersects')


c:\Users\colin\anaconda3\envs\xarray_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
pop_litt['area_flooded'] = pop_litt.area
pop_litt['ratio'] = pop_litt['area_flooded'] / pop_litt['area']

In [69]:
pop_mean_exposed = pop_litt.groupby('lcog_geo').apply(
    lambda g: pd.Series({
        'ind': np.sum(g['ind'] * g['ratio']),
        'ind_65_79': np.sum(g['ind_65_79'] * g['ratio']),
        'ind_80p': np.sum(g['ind_80p'] * g['ratio']),
        'ind_snv': np.sum(g['ind_snv'] * g['ratio']),
    })
).reset_index()
pop_mean_exposed['ind_snv'] = pop_mean_exposed['ind_snv'] / pop_mean_exposed['ind']

C:\Users\colin\AppData\Local\Temp\ipykernel_24420\1183468396.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pop_mean_exposed = pop_litt.groupby('lcog_geo').apply(


In [71]:
pop_mean_exposed, pop_mean_com

(   lcog_geo     ind  ind_65_79  ind_80p       ind_snv
 0     35078   968.0      265.7    103.9  22763.213430
 1     35132  1369.5      253.1     92.3  22589.416867
 2     35186  1019.0      180.8     64.3  22887.249264
 3     35255   910.5      154.6     57.0  21403.368479
 4     80303   392.0       70.6     31.5  23652.818112
 5     85018     4.0        1.0      0.2  23827.100000
 6     85029  1984.5      471.6    158.3  22035.521139
 7     85049  1803.5      324.5     90.4  20547.628944
 8     85104   849.5      223.5     78.5  21899.716775
 9     85267   754.5      112.6     32.1  22234.522333
 10    85297  1005.0      195.6     70.5  21840.703881,
     lcog_geo      ind  ind_65_79  ind_80p       ind_snv
 0      06004  76311.0    13397.7   6845.8  25280.133409
 1      06011   3245.5      571.7    328.8  25477.735973
 2      06027  44965.0     8773.4   4452.8  24723.270742
 3      06029  67438.5    13252.5   7386.4  23119.431449
 4      06032   4230.5      562.4    270.7  28010.4869